# SLV_TRANSFORM_AUTH
**Layer:** Silver  
**Purpose:** Parse and enrich Bronze `brz_auth_logs` → upsert to `slv_auth_logs` → compute per-user auth-failure velocity (1-hour window) and flag high-velocity users.

## 1. Parameters

In [ ]:
batch_id            = "dev-run-00000000"
storage_account     = "adlsbankingdev"
bronze_container    = "bronze"
silver_container    = "silver"
keyvault_name       = "kv-banking-dev"
watermark_date      = "2024-01-01"
run_date            = "2024-01-02"
velocity_threshold  = 5     # failures per user per hour to flag as HIGH_VELOCITY
dq_reject_threshold = 0.05

## 2. Imports and Spark Configuration

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    LongType, BooleanType, TimestampType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import datetime

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "64")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

def adls_path(container, *parts):
    base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    return "/".join([base] + list(parts))

ingestion_timestamp = datetime.datetime.utcnow().isoformat() + "Z"
print(f"batch_id={batch_id}  watermark={watermark_date}  run_date={run_date}")

## 3. Read Bronze Auth Logs — Incremental Window

In [ ]:
bronze_path = adls_path(bronze_container, "raw", "brz_auth_logs")

brz_df = (
    spark.read
    .format("parquet")
    .load(bronze_path)
    .filter(
        (F.col("ingestion_date") >= F.lit(watermark_date)) &
        (F.col("ingestion_date") <  F.lit(run_date))       &
        (F.col("source_system")  == F.lit("auth_logs"))
    )
)
raw_count = brz_df.cache().count()
print(f"Bronze auth records in window [{watermark_date}, {run_date}): {raw_count:,}")

## 4. Parse JSON Raw Payload

In [ ]:
auth_raw_schema = StructType([
    StructField("raw_log_id",       StringType(),  False),
    StructField("user_id",          StringType(),  True),
    StructField("account_id",       StringType(),  True),
    StructField("channel_code",     StringType(),  True),
    StructField("auth_method",      StringType(),  True),   # PASSWORD, OTP, BIOMETRIC
    StructField("auth_result",      StringType(),  True),   # SUCCESS, FAILURE, LOCKED
    StructField("ip_address",       StringType(),  True),
    StructField("user_agent",       StringType(),  True),
    StructField("session_id",       StringType(),  True),
    StructField("device_id",        StringType(),  True),
    StructField("failure_reason",   StringType(),  True),
    StructField("event_timestamp",  StringType(),  True),
])

parsed_df = (
    brz_df
    .withColumn("payload", F.from_json(F.col("raw_payload"), auth_raw_schema))
    .select("payload.*", "ingestion_date", "enqueued_time", "batch_id")
)

## 5. Data Quality Checks

In [ ]:
mandatory_fields = ["raw_log_id", "user_id", "auth_result", "event_timestamp", "ip_address"]
null_cond = None
for f in mandatory_fields:
    c = F.col(f).isNull()
    null_cond = c if null_cond is None else null_cond | c

null_df     = parsed_df.filter(null_cond)
non_null_df = parsed_df.filter(~null_cond)
null_count  = null_df.count()

# Dedup on raw_log_id
w_dedup = Window.partitionBy("raw_log_id").orderBy(F.col("enqueued_time").asc())
deduped_df = (
    non_null_df
    .withColumn("_rn", F.row_number().over(w_dedup))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)
dup_count = null_count + (non_null_df.count() - deduped_df.count())
rejected_count   = null_count + dup_count
reject_fraction  = rejected_count / raw_count if raw_count > 0 else 0.0

print(f"DQ: raw={raw_count:,}  null={null_count:,}  dup_approx={dup_count:,}  "
      f"rejected={rejected_count:,}  reject_pct={reject_fraction*100:.2f}%")
if reject_fraction > dq_reject_threshold:
    raise RuntimeError(f"DQ threshold exceeded: {reject_fraction*100:.2f}% > {dq_reject_threshold*100:.0f}%")

## 6. Derive country_code from IP Address

In [ ]:
# ---------------------------------------------------------------------------
# Placeholder country-code lookup.
# In production, replace with a GeoIP UDF backed by a MaxMind database table
# or an Azure Maps call, reading the lookup Delta table via broadcast join.
# ---------------------------------------------------------------------------
ip_country_map = {
    "10.":  "INTERNAL",
    "192.168.": "INTERNAL",
}

@F.udf(returnType=StringType())
def ip_to_country_code(ip: str) -> str:
    """
    Stub implementation: classifies RFC-1918 ranges as INTERNAL.
    Replace body with broadcast-join against gld_dim_geo_ip at scale.
    """
    if ip is None:
        return "UNKNOWN"
    for prefix, code in [("10.", "INTERNAL"), ("192.168.", "INTERNAL"), ("172.", "INTERNAL")]:
        if ip.startswith(prefix):
            return code
    # Return first two octets as a placeholder region key
    parts = ip.split(".")
    return f"REGION_{parts[0]}_{parts[1]}" if len(parts) >= 2 else "UNKNOWN"

enriched_ip_df = deduped_df.withColumn("country_code", ip_to_country_code(F.col("ip_address")))

## 7. Map to Silver Schema

In [ ]:
auth_result_map = F.create_map(
    F.lit("SUCCESS"), F.lit("SUCCESS"),
    F.lit("S"),       F.lit("SUCCESS"),
    F.lit("FAILURE"), F.lit("FAILURE"),
    F.lit("F"),       F.lit("FAILURE"),
    F.lit("FAIL"),    F.lit("FAILURE"),
    F.lit("LOCKED"),  F.lit("LOCKED"),
    F.lit("L"),       F.lit("LOCKED"),
)

silver_df = (
    enriched_ip_df
    .withColumn("event_timestamp",
        F.to_timestamp(F.col("event_timestamp")))
    .withColumn("event_date_sk",
        F.date_format(F.col("event_timestamp"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("auth_result_std",
        F.coalesce(auth_result_map[F.upper(F.col("auth_result"))], F.upper(F.col("auth_result"))))
    .withColumn("is_failure",
        (F.col("auth_result_std") == F.lit("FAILURE")).cast(BooleanType()))
    .withColumn("slv_batch_id",            F.lit(batch_id))
    .withColumn("slv_ingestion_timestamp", F.lit(ingestion_timestamp))
    .withColumnRenamed("auth_result_std",  "auth_result")
    .drop("ingestion_date")
)

## 8. Compute Auth Failure Velocity (1-Hour Window)

In [ ]:
# ── Rolling count of failures per user in a 1-hour look-back ─────────────────
# Approach: for each event, count FAILURE rows for the same user where
# event_timestamp falls within (current - 1h, current].
# Implemented as a range-based window on epoch seconds.
# ---------------------------------------------------------------------------
failures_only = silver_df.filter(F.col("is_failure") == True)

epoch_window = (
    Window
    .partitionBy("user_id")
    .orderBy(F.col("event_timestamp").cast("long"))
    .rangeBetween(-3600, 0)   # 3600 seconds = 1 hour
)

velocity_df = (
    silver_df
    .withColumn("failure_count_1h",
        F.sum(F.col("is_failure").cast(LongType())).over(epoch_window))
    .withColumn("is_high_velocity",
        (F.col("failure_count_1h") >= F.lit(velocity_threshold)).cast(BooleanType()))
)
high_vel_count = velocity_df.filter(F.col("is_high_velocity") == True).count()
print(f"High-velocity users flagged (>= {velocity_threshold} failures/hour): {high_vel_count:,}")

## 9. Upsert to Silver Delta Table

In [ ]:
silver_table_path = adls_path(silver_container, "delta", "slv_auth_logs")

if not DeltaTable.isDeltaTable(spark, silver_table_path):
    print("Target Delta table not found — creating from current batch.")
    (
        velocity_df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("event_date_sk")
        .save(silver_table_path)
    )
    print(f"Created: {silver_table_path}")
else:
    auth_delta = DeltaTable.forPath(spark, silver_table_path)
    (
        auth_delta.alias("target")
        .merge(
            velocity_df.alias("source"),
            "target.raw_log_id = source.raw_log_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    metrics = auth_delta.history(1).select("operationMetrics").collect()[0][0]
    print(f"MERGE complete. Metrics: {metrics}")

## 10. Log DQ Metrics to Control Table

In [ ]:
from pyspark.sql.types import DoubleType

dq_row = [(
    batch_id,
    "slv_transform_auth",
    "brz_auth_logs",
    "slv_auth_logs",
    ingestion_timestamp,
    int(raw_count),
    int(null_count),
    int(dup_count),
    int(rejected_count),
    int(velocity_df.count()),
    float(reject_fraction),
)]

dq_schema = (
    "batch_id STRING, pipeline_stage STRING, source_table STRING, target_table STRING, "
    "run_ts STRING, raw_count LONG, null_count LONG, duplicate_count LONG, "
    "rejected_count LONG, loaded_count LONG, reject_fraction DOUBLE"
)
dq_df = spark.createDataFrame(dq_row, schema=dq_schema)

ctrl_path = adls_path(silver_container, "delta", "slv_ctrl_dq_metrics")
dq_df.write.format("delta").mode("append").save(ctrl_path)
print(f"DQ metrics appended to {ctrl_path}")
print(f"batch complete: batch_id={batch_id}")